# Análisis de Encuesta — Tecate Pa'l Norte 2026

**Muestra:** 20,277 respuestas  
**Periodo:** Marzo–Abril 2026  

Este notebook analiza las respuestas de la encuesta post-festival:
calificaciones de experiencia, perfil del asistente, escenarios favoritos,
percepción de marcas y NPS.

In [ ]:
# Instala las dependencias en el Python que usa este kernel
import sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "matplotlib", "pandas", "numpy", "--quiet"],
    check=True
)
print("Kernel:", sys.executable)

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("datos.py")))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from datos import cargar_csv

df = cargar_csv("datos.csv")

print(f"Respuestas cargadas : {len(df):,}")
print(f"Columnas            : {len(df.columns)}")
print(f"Periodo             : {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")
df[["nombre", "genero", "estado", "tipo_boleto", "evaluacion_general", "nps"]].head()

## 1. Perfil del asistente

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Perfil del asistente', fontsize=14, fontweight='bold')

# Género
genero_counts = df['genero'].value_counts()
axes[0].pie(genero_counts, labels=genero_counts.index, autopct='%1.0f%%',
            colors=['#4C9BE8', '#E87D7D'], startangle=90)
axes[0].set_title('Género')

# Tipo de boleto
boleto_counts = df['tipo_boleto'].value_counts()
axes[1].bar(boleto_counts.index, boleto_counts.values, color=['#5DBB8A', '#F5A623'])
axes[1].set_title('Tipo de boleto')
axes[1].set_ylabel('Respuestas')
axes[1].tick_params(axis='x', rotation=15)

# Festivales por año
fest_counts = df['festivales_anuales'].value_counts()
axes[2].bar(fest_counts.index, fest_counts.values, color='#9B59B6')
axes[2].set_title('Festivales/conciertos al año')
axes[2].set_ylabel('Respuestas')

plt.tight_layout()
plt.show()

## 2. Calificaciones de experiencia

Comparación de todas las dimensiones calificadas del 1 al 10.

In [ ]:
dimensiones = {
    'Zona': 'evaluacion_zona',
    'Line-up': 'evaluacion_lineup',
    'Seguridad': 'evaluacion_seguridad',
    'Limpieza': 'evaluacion_limpieza',
    'Restaurantes': 'evaluacion_restaurantes',
    'Merch': 'evaluacion_merch',
    'App': 'evaluacion_app',
    'General': 'evaluacion_general',
}

promedios = pd.Series({label: df[col].mean() for label, col in dimensiones.items()}).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(promedios.index, promedios.values,
               color=['#E74C3C' if v < 7 else '#F39C12' if v < 8.5 else '#2ECC71'
                      for v in promedios.values])
ax.axvline(x=7, color='gray', linestyle='--', linewidth=1, label='Umbral 7')
ax.axvline(x=promedios.mean(), color='steelblue', linestyle=':', linewidth=1.5,
           label=f'Promedio global ({promedios.mean():.1f})')

for bar, val in zip(bars, promedios.values):
    ax.text(val + 0.05, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}', va='center', fontsize=9)

ax.set_xlim(0, 11)
ax.set_xlabel('Calificación promedio (1-10)')
ax.set_title(f'Calificaciones promedio por dimensión  (n = {len(df):,})')
ax.legend()
plt.tight_layout()
plt.show()

print("\nRanking de dimensiones:")
for dim, val in promedios.sort_values(ascending=False).items():
    bar = '█' * int(val)
    print(f"  {dim:<14} {val:.2f}  {bar}")

## 3. Radar de experiencia — promedio del grupo

El radar muestra el promedio de las 20,277 respuestas en cada dimensión.

In [ ]:
categorias = list(dimensiones.keys())
N = len(categorias)
angulos = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angulos += angulos[:1]

vals_grupo = [df[col].mean() for col in dimensiones.values()]
vals_grupo += vals_grupo[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.plot(angulos, vals_grupo, color='#3498DB', linewidth=2.5)
ax.fill(angulos, vals_grupo, color='#3498DB', alpha=0.2)

# Añadir referencia del umbral 7
vals_ref = [7] * N + [7]
ax.plot(angulos, vals_ref, color='gray', linewidth=1, linestyle='--', label='Umbral 7')

ax.set_xticks(angulos[:-1])
ax.set_xticklabels(categorias, fontsize=10)
ax.set_ylim(0, 10)
ax.set_yticks([2, 4, 6, 8, 10])
ax.set_yticklabels(['2', '4', '6', '8', '10'], fontsize=7, color='gray')
ax.set_title(f'Radar de experiencia — promedio del grupo  (n = {len(df):,})',
             fontsize=12, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

# Anotar cada punta con su valor
for angulo, val, label in zip(angulos[:-1], vals_grupo[:-1], categorias):
    ax.annotate(f'{val:.1f}', xy=(angulo, val), fontsize=8, color='#2C3E50',
                ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 4. NPS — Net Promoter Score

- **Promotores** (9-10): recomendarían el festival  
- **Pasivos** (7-8): satisfechos pero no entusiastas  
- **Detractores** (0-6): en riesgo de no regresar  

`NPS = (% Promotores − % Detractores) × 100`

In [ ]:
def clasificar_nps(score):
    if score >= 9:
        return 'Promotor'
    elif score >= 7:
        return 'Pasivo'
    else:
        return 'Detractor'

df['categoria_nps'] = df['nps'].apply(clasificar_nps)
nps_counts = df['categoria_nps'].value_counts()

total = len(df)
promotores = (df['categoria_nps'] == 'Promotor').sum()
detractores = (df['categoria_nps'] == 'Detractor').sum()
nps_score = (promotores - detractores) / total * 100

color_map = {'Promotor': '#2ECC71', 'Pasivo': '#F39C12', 'Detractor': '#E74C3C'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribución Promotor/Pasivo/Detractor
order = ['Promotor', 'Pasivo', 'Detractor']
counts = [nps_counts.get(c, 0) for c in order]
bars = axes[0].bar(order, counts, color=[color_map[c] for c in order])
for bar, count in zip(bars, counts):
    pct = count / total * 100
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                 f'{count:,}\n({pct:.1f}%)', ha='center', fontsize=9)
axes[0].set_title('Distribución NPS por categoría')
axes[0].set_ylabel('Respuestas')
axes[0].set_ylim(0, max(counts) * 1.15)

# Histograma de puntajes NPS del 0-10
nps_vals = df['nps'].dropna().astype(int)
score_counts = nps_vals.value_counts().sort_index()
bar_colors = [color_map['Detractor']] * 7 + [color_map['Pasivo']] * 2 + [color_map['Promotor']] * 2
axes[1].bar(score_counts.index, score_counts.values,
            color=[color_map['Detractor'] if s <= 6 else color_map['Pasivo'] if s <= 8
                   else color_map['Promotor'] for s in score_counts.index])
axes[1].set_xticks(range(0, 11))
axes[1].set_title('Distribución de puntajes NPS (0-10)')
axes[1].set_xlabel('Puntaje')
axes[1].set_ylabel('Respuestas')

plt.suptitle(f'NPS del festival: {nps_score:.0f}  '
             f'(Promotores: {promotores:,} | Pasivos: {nps_counts.get("Pasivo",0):,} | '
             f'Detractores: {detractores:,})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"NPS = ({promotores:,} promotores - {detractores:,} detractores) / {total:,} total × 100 = {nps_score:.1f}")

## 5. Escenarios favoritos y género preferido para 2027

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Escenarios favoritos — separados por ";" en el CSV real
escenarios = (
    df['escenario_favorito']
    .dropna()
    .str.split(';')
    .explode()
    .str.strip()
    .value_counts()
    .head(10)
)
axes[0].barh(escenarios.index[::-1], escenarios.values[::-1], color='#3498DB')
axes[0].set_title(f'Top escenarios favoritos  (n = {len(df):,})')
axes[0].set_xlabel('Menciones')
for i, (idx, val) in enumerate(zip(escenarios.index[::-1], escenarios.values[::-1])):
    axes[0].text(val + 20, i, f'{val:,}', va='center', fontsize=8)

# Género para 2027 — top 8 géneros más mencionados
genero_2027 = df['genero_2027'].dropna().value_counts().head(8)
axes[1].bar(genero_2027.index, genero_2027.values,
            color=plt.cm.Set2.colors[:len(genero_2027)])
axes[1].set_title('Género musical que no puede faltar en 2027')
axes[1].set_ylabel('Menciones')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

## 6. Percepción de precios y transporte

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Percepción de precios
precios = df['percepcion_precios'].value_counts()
price_colors = {'Muy Altos': '#E74C3C', 'Altos': '#E67E22', 'Justos': '#2ECC71'}
colors = [price_colors.get(p, 'gray') for p in precios.index]
axes[0].bar(precios.index, precios.values, color=colors)
axes[0].set_title('Percepción de precios en restaurantes')
axes[0].set_ylabel('Asistentes')

# Cómo llegaron al festival
transporte = df['llegada_festival'].value_counts()
axes[1].barh(transporte.index, transporte.values, color='#1ABC9C')
axes[1].set_title('Medio de llegada al festival')
axes[1].set_xlabel('Asistentes')

plt.tight_layout()
plt.show()

## 7. Marcas recordadas como patrocinadores

In [ ]:
# Marcas separadas por coma en el CSV real
marcas = (
    df['marcas_recordadas']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
    .str.title()
    .value_counts()
    .head(20)
)

threshold = marcas.quantile(0.75)  # top cuartil como umbral de alta recordación
bar_colors = ['#E8C33C' if v >= threshold else '#AAAAAA' for v in marcas.values]

plt.figure(figsize=(12, 5))
bars = plt.bar(marcas.index, marcas.values, color=bar_colors)
plt.title(f'Top 20 marcas recordadas como patrocinadores  (n = {len(df):,})')
plt.ylabel('Menciones')
plt.xticks(rotation=35, ha='right')

gold = mpatches.Patch(color='#E8C33C', label='Alta recordación (top 25%)')
gray = mpatches.Patch(color='#AAAAAA', label='Recordación media')
plt.legend(handles=[gold, gray])
plt.tight_layout()
plt.show()

print(f"\nMarcas con mayor recordación (top 5):")
for marca, menciones in marcas.head(5).items():
    pct = menciones / len(df) * 100
    print(f"  {marca:<20} {menciones:>6,} menciones  ({pct:.1f}% de encuestados)")

## 8. App y página web — adopción y satisfacción

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Adopción: App vs Web
app_descargaron = df['descargo_app'].sum()
web_visitaron = df['visito_web'].sum()
total_resp = len(df)

canales = ['App', 'Página web']
si_counts = [app_descargaron, web_visitaron]
no_counts = [total_resp - app_descargaron, total_resp - web_visitaron]

x = np.arange(2)
axes[0].bar(x - 0.2, si_counts, 0.35, label='Sí usaron', color='#3498DB')
axes[0].bar(x + 0.2, no_counts, 0.35, label='No usaron', color='#BDC3C7')
for i, (s, n) in enumerate(zip(si_counts, no_counts)):
    axes[0].text(i - 0.2, s + 50, f'{s/total_resp*100:.0f}%', ha='center', fontsize=8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(canales)
axes[0].set_title(f'Adopción digital  (n = {total_resp:,})')
axes[0].set_ylabel('Respuestas')
axes[0].legend()

# Histograma de calificación de la App
app_scores = df['evaluacion_app'].dropna().astype(int)
score_dist = app_scores.value_counts().sort_index()
bar_c = ['#2ECC71' if s >= 9 else '#F39C12' if s >= 7 else '#E74C3C'
         for s in score_dist.index]
axes[1].bar(score_dist.index, score_dist.values, color=bar_c)
axes[1].axvline(x=app_scores.mean(), color='steelblue', linestyle='--',
                label=f'Promedio: {app_scores.mean():.2f}')
axes[1].set_xticks(range(1, 11))
axes[1].set_title('Distribución de calificación de la App')
axes[1].set_xlabel('Calificación (1-10)')
axes[1].set_ylabel('Respuestas')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Resumen ejecutivo de insights

In [ ]:
print("=" * 62)
print("  RESUMEN EJECUTIVO — Encuesta Tecate Pa'l Norte 2026")
print("=" * 62)

top_estado = df['estado'].value_counts().head(3).to_dict()
top_genero = df['genero'].value_counts().to_dict()
volveria_si = (df['volveria_asistir'].str.contains('definitivamente', na=False)).sum()
top_marcas  = marcas.head(3).index.tolist()
top_escenario = escenarios.index[0] if len(escenarios) else 'N/D'
top_genero27  = df['genero_2027'].value_counts().index[0] if df['genero_2027'].notna().any() else 'N/D'

print(f"""
MUESTRA
  Total respuestas    : {len(df):,}
  Géneros             : {top_genero}
  Top 3 estados       : {top_estado}

SATISFACCIÓN PROMEDIO (1-10)
  Experiencia general : {df['evaluacion_general'].mean():.2f}
  Line-up             : {df['evaluacion_lineup'].mean():.2f}
  Seguridad           : {df['evaluacion_seguridad'].mean():.2f}
  Zona                : {df['evaluacion_zona'].mean():.2f}
  Restaurantes        : {df['evaluacion_restaurantes'].mean():.2f}
  App                 : {df['evaluacion_app'].mean():.2f}
  Merch               : {df['evaluacion_merch'].mean():.2f}
  Limpieza sanitarios : {df['evaluacion_limpieza'].mean():.2f}  ← área de mejora

LEALTAD
  NPS                 : {nps_score:.1f}
  Promotores (9-10)   : {promotores:,}  ({promotores/total*100:.1f}%)
  Detractores (0-6)   : {detractores:,}  ({detractores/total*100:.1f}%)
  Volvería definitivamente: {volveria_si:,}  ({volveria_si/total*100:.1f}%)

PREFERENCIAS 2027
  Género más pedido   : {top_genero27}
  Escenario favorito  : {top_escenario}

PRECIOS
  Percepción          : {df['percepcion_precios'].value_counts().to_dict()}

TECNOLOGÍA
  Descargaron la app  : {df['descargo_app'].sum():,}  ({df['descargo_app'].mean()*100:.1f}%)
  Visitaron web       : {df['visito_web'].sum():,}  ({df['visito_web'].mean()*100:.1f}%)

MARCAS MÁS RECORDADAS (top 3)
  {", ".join(top_marcas)}
""")
print("=" * 62)